# 🚗 RoadVision AI — Final Detector Training on RDD2022 India + Local V1
**Hardware:** Google Colab T4 GPU (`Runtime -> Change runtime type -> T4 GPU`)

### Required Files to Upload:
1. `dataset_rdd_india_potholes.zip` (124 MB) -> upload to `/content/`
2. `pothole_best.pt` (24 MB) -> upload to `/content/`

In [ ]:
# 1. Verify NVIDIA T4 GPU Acceleration
!nvidia-smi

In [ ]:
# 2. Install Ultralytics YOLOv8
!pip install -q ultralytics

In [ ]:
# 3. Extract Dataset and Configure data.yaml Path
import yaml

!unzip -q /content/dataset_rdd_india_potholes.zip -d /content/dataset_rdd_india_potholes

yaml_path = '/content/dataset_rdd_india_potholes/data.yaml'
with open(yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = '/content/dataset_rdd_india_potholes'
with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print('✅ Dataset extracted. data.yaml content:')
print(open(yaml_path).read())

In [ ]:
# 4. Launch Fine-Tuning on T4 GPU (25 Epochs, imgsz=640, AdamW, lr0=0.002)
from ultralytics import YOLO

model = YOLO('/content/pothole_best.pt')

results = model.train(
    data='/content/dataset_rdd_india_potholes/data.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    device=0,
    optimizer='AdamW',
    lr0=0.002,
    lrf=0.01,
    project='runs/detect',
    name='pothole_rdd_india',
    patience=10,
    save=True,
    augment=True,
    close_mosaic=10
)

print('🎉 Training complete!')

In [ ]:
# 5. Download Best Trained Weights to Local PC
from google.colab import files
import os

best_weights = '/content/runs/detect/pothole_rdd_india/weights/best.pt'
if os.path.exists(best_weights):
    print(f'Downloading {best_weights}...')
    files.download(best_weights)
else:
    print('Weights file not found. Check training logs above.')